# Validate the FD001 input files.

Recorded-run workflow. Configure paths in `settings.json`; see the root README for dependencies.


In [ ]:
from pathlib import Path
import sys

_release = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "release_runtime.py").is_file()), None)
if _release is None:
    raise FileNotFoundError("Open this notebook from the extracted Code_Release_Candidate folder.")
sys.path.insert(0, str(_release))
from release_runtime import activate_project, load_json, load_text
PROJECT_ROOT, RELEASE_CLASSICAL_PYTHON, RELEASE_TORCH_PYTHON = activate_project()


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import numpy as np
import pandas as pd


DATA_DIR = Path("data")
OUTPUT_DIR = Path("revision_outputs") / "01_FD001_data_audit"
DATASET = "FD001"



EXPECTED_ROWS = {"train": 20631, "test": 13096}
EXPECTED_UNITS = {"train": 100, "test": 100}

ID_COLS = ["UnitNumber", "Time"]
SETTING_COLS = ["OpSet1", "OpSet2", "OpSet3"]
SENSOR_COLS = [f"Sensor{i}" for i in range(1, 22)]
COLUMNS = ID_COLS + SETTING_COLS + SENSOR_COLS

paths = {
    "train": DATA_DIR / "train_FD001.txt",
    "test": DATA_DIR / "test_FD001.txt",
    "rul": DATA_DIR / "RUL_FD001.txt",
}
print("Veri klasörü:", DATA_DIR.resolve())


In [ ]:
def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def read_engine_file(path, split, expected_rows, expected_units):
    raw = pd.read_csv(path, sep=r"\s+", header=None)
    require(raw.shape[1] == len(COLUMNS),
            f"{path.name}: {raw.shape[1]} sütun var; 26 bekleniyor. Dosyayı kontrol edin.")
    raw.columns = COLUMNS
    for column in COLUMNS:
        raw[column] = pd.to_numeric(raw[column], errors="raise")

    values = raw[COLUMNS].to_numpy(dtype=float)
    require(np.isfinite(values).all(),
            f"{path.name}: eksik/sonsuz değer bulundu; satırlar otomatik silinmedi.")
    for column in ID_COLS:
        vals = raw[column].to_numpy()
        require(np.all(vals == np.floor(vals)) and np.all(vals > 0),
                f"{path.name}: {column} pozitif tam sayı olmalı.")
        raw[column] = raw[column].astype("int64")

    require(len(raw) == expected_rows,
            f"{path.name}: {len(raw)} satır var; {expected_rows} bekleniyor.")
    unit_ids = np.sort(raw["UnitNumber"].unique())
    require(np.array_equal(unit_ids, np.arange(1, expected_units + 1)),
            f"{path.name}: motor kimlikleri 1..{expected_units} aralığında olmalı.")
    require(not raw.duplicated(ID_COLS).any(),
            f"{path.name}: tekrarlanan motor/çevrim çifti bulundu.")

    for unit, group in raw.groupby("UnitNumber", sort=False):
        cycles = group["Time"].to_numpy()
        require(np.array_equal(cycles, np.arange(1, len(group) + 1)),
                f"{path.name}: motor {unit} çevrimleri 1'den başlayıp sıralı artmıyor.")

    info = {
        "shape": [int(raw.shape[0]), int(raw.shape[1])],
        "units": int(len(unit_ids)),
        "sensor_count": len(SENSOR_COLS),
        "duplicate_engine_cycle_pairs": 0,
        "nonfinite_values": 0,
        "cycles_start_at_one_and_are_consecutive": True,
    }
    return raw, info


def compare_old_sensor_reading(path, corrected):


    old_columns = ["UnitNumber", "Time", "OpSet1", "OpSet2"] + SENSOR_COLS
    old = pd.read_csv(path, sep=r"\s+", header=None, names=old_columns)
    old_sensors = old[SENSOR_COLS].to_numpy(dtype="float32")
    new_sensors = corrected[SENSOR_COLS].to_numpy(dtype="float32")
    same_shape = old_sensors.shape == new_sensors.shape
    equal = same_shape and np.array_equal(old_sensors, new_sensors)
    maximum_difference = (float(np.max(np.abs(old_sensors - new_sensors)))
                          if same_shape else None)
    return {
        "float32_sensor_values_equal": bool(equal),
        "max_abs_difference": maximum_difference,
        "comparison_dtype": "float32 (eski notebook ile aynı)",
    }


def build_evaluation_metadata(frame, split, terminal_rul=None):
    metadata = frame[ID_COLS].copy()
    metadata.insert(0, "source_row", np.arange(len(frame), dtype="int64"))
    metadata.insert(0, "split", split)
    metadata.insert(0, "dataset", DATASET)
    metadata["engine_key"] = (
        DATASET + ":" + split + ":" + metadata["UnitNumber"].astype(str)
    )
    metadata["sample_key"] = metadata["engine_key"] + ":" + metadata["Time"].astype(str)
    last_observed_cycle = frame.groupby("UnitNumber")["Time"].transform("max")
    rul = last_observed_cycle - frame["Time"]
    if split == "test":
        require(terminal_rul is not None, "Test RUL eşleştirmesi eksik.")
        offsets = frame["UnitNumber"].map(terminal_rul)
        require(offsets.notna().all(), "Bazı test motorlarının son çevrim RUL değeri eksik.")
        rul = rul + offsets
    metadata["RUL"] = rul.to_numpy(dtype="int64")
    require(metadata["sample_key"].is_unique, "Örnek anahtarları benzersiz değil.")
    require((metadata["RUL"] >= 0).all(), "Negatif RUL oluştu.")
    for _, group in metadata.groupby("engine_key", sort=False):
        require(np.all(np.diff(group["RUL"].to_numpy()) == -1),
                "RUL çevrim başına bir azalmıyor.")
    return metadata


In [ ]:
missing = [str(path) for path in paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Eksik veri dosyaları:\n" + "\n".join(missing)
        + "\nDATA_DIR yolunu düzeltin veya üç FD001 dosyasını bu klasöre koyun."
    )

train_df, train_info = read_engine_file(
    paths["train"], "train", EXPECTED_ROWS["train"], EXPECTED_UNITS["train"]
)
test_df, test_info = read_engine_file(
    paths["test"], "test", EXPECTED_ROWS["test"], EXPECTED_UNITS["test"]
)

print(pd.DataFrame([train_info, test_info], index=["train", "test"]).to_string())
print("\nİlk eğitim satırlarının kimlik ve çalışma ayarları:")
print(train_df[ID_COLS + SETTING_COLS].head().to_string(index=False))


In [ ]:
legacy_check = {
    "train": compare_old_sensor_reading(paths["train"], train_df),
    "test": compare_old_sensor_reading(paths["test"], test_df),
}
print("Eski ve düzeltilmiş okumada sensör değerleri:")
print(json.dumps(legacy_check, ensure_ascii=False, indent=2))

sensor_audit = pd.DataFrame({
    "sensor": SENSOR_COLS,
    "train_unique_values": [int(train_df[c].nunique()) for c in SENSOR_COLS],
    "train_std": [float(train_df[c].std(ddof=0)) for c in SENSOR_COLS],
    "train_min": [float(train_df[c].min()) for c in SENSOR_COLS],
    "train_max": [float(train_df[c].max()) for c in SENSOR_COLS],
})
sensor_audit["constant_in_train"] = sensor_audit["train_unique_values"] == 1
print("\nEğitim verisi sensör dökümü; bu aşamada hiçbir sensör çıkarılmadı:")
print(sensor_audit.to_string(index=False))


In [ ]:
rul_file = pd.read_csv(paths["rul"], sep=r"\s+", header=None)
require(rul_file.shape == (EXPECTED_UNITS["test"], 1),
        f"RUL dosyası {EXPECTED_UNITS['test']} satır ve tek sütun içermeli; "
        f"gelen boyut {rul_file.shape}.")
end_rul = pd.to_numeric(rul_file.iloc[:, 0], errors="raise").to_numpy(dtype=float)
require(np.isfinite(end_rul).all() and np.all(end_rul >= 0)
        and np.all(end_rul == np.floor(end_rul)),
        "Son çevrim RUL değerleri sıfır veya pozitif tam sayı olmalı.")

unit_ids = np.sort(test_df["UnitNumber"].unique())
terminal_rul = pd.Series(end_rul.astype("int64"), index=unit_ids, name="terminal_RUL")
train_metadata = build_evaluation_metadata(train_df, "train")
test_metadata = build_evaluation_metadata(test_df, "test", terminal_rul)

train_end = train_metadata.groupby("UnitNumber")["RUL"].last()
test_end = test_metadata.groupby("UnitNumber")["RUL"].last().reindex(unit_ids)
require((train_end == 0).all(), "Eğitim motorlarının son çevrim RUL değeri sıfır olmalı.")
require(np.array_equal(test_end.to_numpy(), terminal_rul.to_numpy()),
        "Test son çevrim RUL değerleri resmî dosyayla eşleşmiyor.")
require(set(train_metadata["engine_key"]).isdisjoint(test_metadata["engine_key"]),
        "Bölme adını içeren motor anahtarları train/test arasında ayrık olmalı.")



X_train_sensors = train_df[SENSOR_COLS].copy()
X_test_sensors = test_df[SENSOR_COLS].copy()
require(list(X_train_sensors.columns) == SENSOR_COLS, "Girdi sensör listesi değişti.")
require("RUL" not in X_train_sensors.columns, "RUL model girdisine karıştı.")
print("RUL eşleştirmesi ve örnek anahtarları doğrulandı.")
print("Girdi matrisleri:", X_train_sensors.shape, X_test_sensors.shape)


In [ ]:
all_sensors_equal = all(
    result["float32_sensor_values_equal"] for result in legacy_check.values()
)
summary = {
    "stage": "01_data_audit",
    "dataset": DATASET,
    "status": "PASS" if all_sensors_equal else "REVIEW_SENSOR_DIFFERENCE",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "versions": {
        "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__
    },
    "train": train_info,
    "test": test_info,
    "legacy_sensor_comparison": legacy_check,
    "constant_sensors_in_train": sensor_audit.loc[
        sensor_audit["constant_in_train"], "sensor"
    ].tolist(),
    "all_21_sensors_retained": True,
    "rul_rows": int(len(end_rul)),
    "train_terminal_rul_zero": True,
    "test_terminal_rul_matches_file": True,
    "rul_is_separate_from_sensor_inputs": True,
    "binary_anomaly_labels_created": False,
    "models_trained": False,
    "file_sha256": {paths[key].name: file_sha256(paths[key]) for key in paths},
    "next_stage": "Motor bazında bölme ve bağımsız değerlendirme protokolünü sabitleme",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary_path = OUTPUT_DIR / "validation_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
sensor_audit.to_csv(OUTPUT_DIR / "sensor_audit.csv", index=False)
train_metadata.to_csv(OUTPUT_DIR / "train_evaluation_metadata.csv", index=False)
test_metadata.to_csv(OUTPUT_DIR / "test_evaluation_metadata.csv", index=False)

print("DATA AUDIT:", summary["status"])
print("Train:", train_info["shape"], "| Motor:", train_info["units"])
print("Test :", test_info["shape"], "| Motor:", test_info["units"])
print("Eski okumayla sensör eşitliği:", all_sensors_equal)
print("RUL satırı:", len(end_rul), "| RUL model girdisinden ayrı: True")
print("\nPaylaşılacak kısa rapor:", summary_path.resolve())
if not all_sensors_equal:
    print("Sensörlerde fark bulundu. Yeni model eğitimine geçmeden bu farkı inceleyin.")
